# Chat with the Book — grounded RAG over an equipment manual

**No fine-tuning. The facts live in the manual, not in the model.**

This notebook turns a 385-page equipment maintenance manual (public-domain U.S.
Army TM 5-3805-237-35, a road grader) into something you can *ask questions of*
in plain English and get **exact, cited answers** back — or an honest "not in the
manual" when the answer isn't there.

The architecture, one line at a time:

```
your question
  -> retrieve the most relevant pages   (classic search — NO model)
  -> a reader LLM answers ONLY from those pages, and cites them
  -> a gate checks the answer is grounded; if not, it abstains
  -> reply + page citation + a trust flag
```

Two things worth noticing as you go:

1. **Retrieval needs no model.** Finding the right spec is a search problem. The
   generative model shows up only as a *reader* that phrases the answer from what
   search already found. Generate for behavior; retrieve for knowledge.
2. **Trust is built in.** Every answer carries a page citation, and the system is
   allowed to say "I don't know" — the two things a fine-tuned model can't do,
   and the two things a regulated shop actually needs.

You can run the reader with **Claude** (needs your Anthropic key) or a **local
gemma** model via Ollama (no key, runs on the Colab GPU).

## 1. Install

In [ ]:
# Core (retrieval) + the two reader backends. ~30s.
!pip -q install pypdf scikit-learn anthropic requests

## 2. Get the manual and build the corpus

We download the public-domain PDF and turn it into page-anchored chunks so every
answer can point back to a page. (You can also `!git clone` your repo instead and
reuse `src/build_corpus.py` — this cell just keeps the notebook self-contained.)

In [ ]:
import os, re, json, urllib.request
from pypdf import PdfReader

PDF = "manual.pdf"
URL = "https://archive.org/download/armytechnicalman003473mbp/armytechnicalman003473mbp.pdf"
if not os.path.exists(PDF):
    print("downloading manual ...")
    urllib.request.urlretrieve(URL, PDF)   # if this ever 404s, upload a PDF and set PDF=<name>

def clean(t):
    t = t.replace("\x00", " ")
    return re.sub(r"[ \t]+", " ", t).strip()

CHUNK, OVERLAP = 700, 120
corpus = []
reader = PdfReader(PDF)
for pageno, page in enumerate(reader.pages, start=1):
    text = clean(page.extract_text() or "")
    if not text:
        continue
    i = 0
    while i < len(text):
        ch = text[i:i+CHUNK]
        if len(ch) >= 40:
            corpus.append({"page": pageno, "text": ch})
        i += CHUNK - OVERLAP

print(f"pages: {len(reader.pages)}   chunks: {len(corpus)}")

## 3. The retriever (search — no model)

Plain TF-IDF over the chunks. This is the part that already scored 12/12 on
finding exact specs in the companion repo, with no LLM involved.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

_texts = [c["text"] for c in corpus]
_vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
_mat = _vec.fit_transform(_texts)

def retrieve(question, k=4):
    sims = cosine_similarity(_vec.transform([question]), _mat)[0]
    idx = sims.argsort()[::-1][:k]
    return [corpus[i] for i in idx]

# quick smoke test
for c in retrieve("valve clearance", k=2):
    print(f"p{c['page']}: {c['text'][:90]}...")

## 4. Pick a reader backend

- **Claude** — set your key (in Colab: 🔑 Secrets → add `ANTHROPIC_API_KEY`, or you'll be prompted).
- **gemma (local, no key)** — installs Ollama and pulls `gemma:2b`. Best with a GPU runtime (Runtime → Change runtime type → GPU).

Run whichever cell(s) you want; you choose per-question in step 6.

In [ ]:
# --- Claude reader -------------------------------------------------------
import os, anthropic

# Cheap, fast reader model. Alternatives: "claude-sonnet-4-6", "claude-opus-4-8".
CLAUDE_MODEL = "claude-haiku-4-5-20251001"

def _get_key():
    try:
        from google.colab import userdata
        k = userdata.get("ANTHROPIC_API_KEY")
        if k:
            return k
    except Exception:
        pass
    k = os.environ.get("ANTHROPIC_API_KEY")
    if not k:
        import getpass
        k = getpass.getpass("ANTHROPIC_API_KEY: ")
    return k

_claude = None
def claude_reader(system, user):
    global _claude
    if _claude is None:
        _claude = anthropic.Anthropic(api_key=_get_key())
    msg = _claude.messages.create(
        model=CLAUDE_MODEL, max_tokens=300, system=system,
        messages=[{"role": "user", "content": user}],
    )
    return msg.content[0].text.strip()

print("Claude reader ready:", CLAUDE_MODEL)

In [ ]:
# --- Local gemma reader via Ollama (no key) ------------------------------
# Run this cell only if you want the keyless local path. First run ~2-3 min.
import subprocess, time, requests

def start_ollama(model="gemma:2b"):
    if subprocess.run("command -v ollama", shell=True).returncode != 0:
        print("installing ollama ...")
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)
    # start server if not already up
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
    except Exception:
        subprocess.Popen(["ollama", "serve"])
        time.sleep(5)
    print(f"pulling {model} ...")
    subprocess.run(["ollama", "pull", model])
    return model

OLLAMA_MODEL = "gemma:2b"   # gemma:2b is light; try "gemma2:9b" on a big GPU

def gemma_reader(system, user):
    prompt = f"{system}\n\n{user}"
    r = requests.post("http://localhost:11434/api/generate",
                      json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False})
    return r.json()["response"].strip()

# Uncomment to set up the local model now:
# start_ollama(OLLAMA_MODEL)
print("gemma reader defined (call start_ollama() to activate)")

## 5. The grounded answer function (with the trust gate)

This is the heart of it. The reader is told to answer **only** from the retrieved
pages and to reply `NOT IN MANUAL` if the answer isn't there. We then:

- **abstain** if it said `NOT IN MANUAL` (no guessing),
- **cite** the pages we actually retrieved,
- flag a light **grounding check**: for spec questions, does a number in the
  answer actually appear in the retrieved source? If not, we downgrade trust.

The grounding signal is based on the *source*, not the model's self-confidence —
which is exactly the signal you can defend to an auditor.

In [ ]:
import re

SYSTEM = (
    "You answer questions about an equipment maintenance manual. "
    "Use ONLY the CONTEXT provided. Quote exact values verbatim. "
    "Cite the page like [p.NN]. If the answer is not in the CONTEXT, "
    "reply with exactly: NOT IN MANUAL"
)

def _nums(s):
    return set(re.findall(r"\d+(?:\.\d+)?(?:/\d+)?", s))

def ask(question, backend="claude", k=4):
    hits = retrieve(question, k=k)
    context = "\n---\n".join(f"[p.{h['page']}] {h['text']}" for h in hits)
    reader = {"claude": claude_reader, "gemma": gemma_reader}[backend]
    reply = reader(SYSTEM, f"CONTEXT:\n{context}\n\nQUESTION: {question}")

    pages = sorted({h["page"] for h in hits})
    if "NOT IN MANUAL" in reply.upper():
        trust = "ABSTAINED (not found — no guess)"
        answer = "I couldn't find that in the manual."
    else:
        # light grounding check: any number in the answer present in the sources?
        ans_nums, ctx_nums = _nums(reply), _nums(context)
        grounded = (not ans_nums) or bool(ans_nums & ctx_nums)
        trust = "GROUNDED ✓" if grounded else "UNVERIFIED — number not found in source ⚠"
        answer = reply
    return {"answer": answer, "trust": trust, "pages": pages}

def show(question, backend="claude"):
    r = ask(question, backend=backend)
    print(f"Q: {question}")
    print(f"A: {r['answer']}")
    print(f"   trust: {r['trust']}   pages searched: {r['pages']}")
    print("-" * 70)

print("ask() ready")

## 6. Try it

In [ ]:
# Uses Claude by default. Pass backend="gemma" after start_ollama().
show("What is the maximum warpage limit?")
show("At what voltage should the variable resistance be set?")
show("At what brush length should the brushes be replaced?")
show("What is the air gap between the armature and coil?")
show("What is the recommended tire pressure for a bicycle?")  # not in the manual -> should abstain

## 7. Chat with the book (terminal-style loop)

In [ ]:
# Type a question and press Enter. Empty line to quit.
BACKEND = "claude"   # or "gemma" once start_ollama() has run
while True:
    try:
        q = input("\nyou > ").strip()
    except (EOFError, KeyboardInterrupt):
        break
    if not q:
        print("bye")
        break
    r = ask(q, backend=BACKEND)
    print(f"book> {r['answer']}")
    print(f"      [{r['trust']}]  pages: {r['pages']}")

## What this demo is (and isn't) proving

- The **knowledge** came from retrieval — a search index over the manual, no model
  required to *find* the fact. Swap the PDF for your own manual and nothing changes.
- The **reader** (Claude or gemma) only *phrases* the grounded answer and cites it.
  That's behavior, not knowledge.
- The **gate** makes it trustworthy: it abstains instead of inventing, and the
  trust flag is tied to the source, not the model's mood.

This is the honest, auditable shape of "chat with our docs" — the opposite of
fine-tuning a model and hoping it memorized the numbers.

*Manual: public-domain U.S. Army TM 5-3805-237-35. Companion repo:
`finetune-vs-retrieve-demo`.*